# 동아대 대학원 QML 강의 — 10월 준비 (2일차용)
## EDA + 데이터 파이프라인 리허설 (Colab 전용)

**목적**: 11/20 2일차 수업에서 학생들이 라이브로 진행할 작업(kagglehub 다운로드 → EDA → 세그멘테이션
전처리 → 회로 인코딩 기초)을 강사가 **미리 한 번 끝까지 돌려보는** 리허설 노트북입니다.
2일차는 학습(training)이 없는 시간이라 체크포인트를 만들 필요가 없고, 이 노트북의 목적도
"산출물 생성"이 아니라 **"수업 당일 문제없이 진행되는지 확인"**입니다.

**왜 로컬이 아니라 Colab인가요?** 로컬 PC에는 Kaggle API가 설정되어 있지 않아 HAM10000을
받을 수 없습니다. 실제 수업 환경(Colab)에서 리허설해야 당일 발생할 수 있는 문제(다운로드 속도,
설치 오류, 세그멘테이션 소요시간 등)를 미리 잡을 수 있습니다.

**2일차 학생용 노트북과의 관계**: 이 노트북은 **학생에게 배포하지 않습니다.**
`day2/day2_practice.ipynb`(학생용)와 **완전히 동일한 코드**를 그대로
실행해보는 것이 목적이며, 여기서 발견된 문제는 학생용 노트북에 반영합니다.

**추가로 하는 일**: 전처리된 DF/NV 이미지를 Google Drive에 저장해두면, 당일 Kaggle 접속이
불안정할 때 학생들이 바로 불러다 쓸 수 있는 **폴백(fallback) 데이터**로도 씁니다(선택 사항).

**3일차 준비와는 별개입니다** — QGAN/CNN 체크포인트 생성은
`day3/prep_day3_train_checkpoints.ipynb`에서 진행합니다.


## 1. 환경 설정

In [ ]:
!pip install -q pennylane pennylane-lightning scikit-fuzzy kagglehub koreanize-matplotlib
print("설치 완료")

In [ ]:
import os, math
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
import pennylane as qml
import skfuzzy as fuzz
from sklearn.cluster import KMeans

# 그래프 한글 표시 — Colab 기본 폰트에는 한글이 없어 설정하지 않으면 그래프 글자가 네모(□)로 깨짐
try:
    import koreanize_matplotlib  # 나눔고딕 폰트 등록 + 기본 폰트 지정
except Exception as e:
    print("한글 폰트 설정 실패 — 그래프의 한글만 깨지고 실습 진행에는 지장 없음:", e)

print("torch:", torch.__version__, "| pennylane:", qml.__version__, "| opencv:", cv2.__version__)

dev = qml.device("lightning.qubit", wires=1)
@qml.qnode(dev)
def _sanity_check():
    qml.Hadamard(wires=0)
    return qml.expval(qml.PauliZ(0))
print("PennyLane 동작 확인 (0에 가까운 값이 나와야 정상):", _sanity_check())

## 2. HAM10000 다운로드 (kagglehub)

2일차 학생용 노트북과 동일하게 `kagglehub`로 다운로드함(약 6GB, Colab 런타임 디스크에 저장).
HAM10000은 공개 데이터셋이라 로그인 없이 다운로드됨(학생용 노트북도 로그인 불필요로 안내함) — 로그인 셀은
다운로드에서 인증 오류가 날 때만 사용. Colab에서는 Kaggle 전용 캐시에서 바로 연결(`/kaggle/input/...`)되어
수 초 만에 끝날 수 있음 — 아래 소요시간 출력으로 어느 쪽인지 확인할 것.

In [ ]:
import kagglehub
# (선택) 인증 오류 시에만: 주석 해제 → 토큰 입력 → Login 클릭 → 성공 메시지 확인 후 다음 셀 실행
# kagglehub.login()

In [ ]:
import kagglehub
import time as _t
_t0 = _t.time()
DATA_DIR = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print(f"다운로드 소요시간: {(_t.time()-_t0)/60:.1f}분 (수업 당일 학생 다운로드 시간 배분 참고) | 경로: {DATA_DIR}")

metadata = pd.read_csv(f"{DATA_DIR}/HAM10000_metadata.csv")
print(f"전체 이미지 수: {len(metadata)}")
metadata.head()

> **강사용 메모**: 아래는 2일차 학생용 노트북의 다운로드 셀 바로 뒤에 들어간 설명과 **동일한 내용**임. 수업 당일 다운로드를 기다리는 동안 이 흐름대로 설명하며, 리허설 시 위 셀의 다운로드 소요시간을 보고 설명 분량(질문 토론 포함 여부)을 조절할 것.

### ⏳ 다운로드 대기 중 — 오늘 다룰 EDA·전처리 파이프라인 개요

위 다운로드 셀은 수 분이 걸릴 수 있음. 셀이 실행되는 동안 아래 내용으로 오늘 실습 전체 흐름을 먼저 짚어봄.
(셀 왼쪽 실행 표시가 멈추고 `데이터셋 경로:`가 출력되면 다음 셀로 진행함)

#### 전체 흐름 한눈에 보기

```
원본 이미지 (600×450 RGB, 10,015장)
   │ ① EDA: 클래스 분포·샘플 확인 → DF vs NV 이진분류로 범위 축소
   ▼
DF 115장 / NV 6,705장 (약 1:58 불균형)
   │ ② 세그멘테이션: grayscale → 64×64 축소 → K-means → Fuzzy C-means → 병변만 남김
   ▼
병변만 남은 64×64 grayscale
   │ ③ 초저해상도 축소: 64×64 → 8×8 (0~1 정규화)
   ▼
8×8 = 64픽셀 벡터
   │ ④ 양자 인코딩: 64 = 2⁶ → 데이터 큐빗 6개의 확률분포로 표현
   ▼
서브제너레이터 회로 (오늘은 forward만 확인, 학습은 3일차에 사전학습본 사용)
```

#### 단계별 핵심

**① EDA (2번 섹션)**
- HAM10000은 7개 클래스로 구성되나 극단적으로 불균형함 — NV(모반)가 전체의 약 67%, DF(피부섬유종)는 약 1.1%에 불과함.
- 논문(Andra et al.)과 동일하게 **최소 클래스 DF vs 최다 클래스 NV** 조합을 선택함. 불균형이 가장 심한 조합이라 "생성 모델로 소수 클래스를 증강하면 분류가 나아지는가"라는 질문을 검증하기에 적합함.
- 확인할 것: 클래스별 이미지 수(막대그래프), 클래스별 샘플 이미지의 시각적 차이.

**② 세그멘테이션 (3번 섹션) — 파이프라인의 핵심 단계**
- 논문 Table 10에서 QGAN 증강은 **원본(raw) 이미지에서는 오히려 성능을 떨어뜨렸고, 세그멘테이션을 거친 이미지에서만 개선됨.**
- 이유: 큐빗 수 제약으로 이미지를 8×8까지 줄여야 함. 배경(피부·털·조명 얼룩)이 섞인 채로 줄이면 64픽셀 안에 병변 정보가 거의 남지 않음. 병변만 남겨두고 줄여야 적은 픽셀로도 "병변의 모양·위치" 정보가 보존됨.
- 방법: 밝기값만으로 픽셀을 4개 그룹으로 나눔 → K-means로 대략 나눈 결과를 출발점으로 Fuzzy C-means(각 픽셀이 여러 그룹에 부분적으로 속하는 방식)가 경계를 정제함 → **가장 어두운 그룹 = 병변**으로 보고 나머지는 0(검정)으로 지움.
- 한계(직접 확인해볼 것): 피부경 이미지 모서리의 검은 테두리(비네팅)가 병변보다 어두우면 테두리가 병변으로 잘못 선택될 수 있음. 실제 결과 이미지에서 이런 사례가 보이는지 관찰함.

**③ 초저해상도 축소 (3번 섹션 후반)**
- 64×64 → 8×8로 축소하고 0~1로 정규화함. 사람이 보기엔 거의 형체만 남지만, 이 수준이 현재 시뮬레이터로 다룰 수 있는 규모임.
- `IMG_SIZE`를 4로 바꾸면 4×4=16픽셀(데이터 큐빗 4개)로 더 작게 실습 가능함.

**④ 양자 인코딩 + 회로 (4번 섹션)**
- 64픽셀 = 2⁶ 이므로 **데이터 큐빗 6개의 측정 확률분포(64개 값)를 이미지 한 장으로 해석**함 — 큐빗 수가 1개 늘 때마다 표현 가능한 픽셀 수가 2배가 됨.
- 오늘은 랜덤 파라미터로 회로가 정상 동작하는지(출력 64개, 확률 합 1)만 확인함. 3일차에는 이 회로를 여러 개 이어붙인(patch 방식) 사전학습 QGAN을 불러와 실제 생성 결과를 확인함.

#### 생각해볼 질문 (다운로드를 기다리며)
1. 불균형 1:58 상태로 분류기를 학습하면 어떤 문제가 생기는가? (힌트: 전부 NV라고 답해도 정확도 98%)
2. 8×8까지 줄인 이미지로도 DF와 NV를 구분할 수 있을까? 어떤 정보가 남고 어떤 정보가 사라지는가?
3. 세그멘테이션을 하지 않고 원본을 바로 8×8로 줄이면 어떤 이미지가 될지 예상해봄.

## 3. EDA — 클래스 분포 + 샘플 이미지

2일차 학생용 노트북 2번 섹션과 동일. 논문 Fig.12와 동일한 클래스 불균형을 확인합니다.

In [ ]:
class_names_kr = {"akiec": "광선각화증", "bcc": "기저세포암", "bkl": "양성각화증",
                   "df": "피부섬유종", "mel": "흑색종", "nv": "모반(정상)", "vasc": "혈관병변"}
counts = metadata["dx"].value_counts()
plt.figure(figsize=(8, 4))
plt.bar([class_names_kr[c] for c in counts.index], counts.values)
plt.ylabel("이미지 수"); plt.title("HAM10000 클래스별 분포")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
print(counts)
print(f"\nDF(피부섬유종): {counts.get('df', 0)}장 vs NV(모반): {counts.get('nv', 0)}장",
      f"— 불균형 비율 약 1:{counts.get('nv',0)//max(1,counts.get('df',1))}")

In [ ]:
def find_image_path(image_id):
    for sub in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        p = f"{DATA_DIR}/{sub}/{image_id}.jpg"
        if os.path.exists(p):
            return p
    return None

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, cls in zip(axes.flat, list(class_names_kr.keys()) + ["df"]):
    row = metadata[metadata["dx"] == cls].iloc[0]
    path = find_image_path(row["image_id"])
    if path:
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.set_title(f"{cls} ({class_names_kr[cls]})"); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. 세그멘테이션 파이프라인 (2일차 학생용 노트북과 동일 로직)

`day3/data_pipeline.py`와 완전히 동일한 함수입니다. 여기서 소요 시간·에러 여부를
미리 확인해 수업 당일 시간 배분(전처리 실습 30분)이 현실적인지 점검합니다.

In [ ]:
def segment_fcm_kmeans(img_bgr, n_clusters=4, resize_to=None):
    """HAM10000 QGAN 논문 Section 3.1 — K-means+Fuzzy C-means 하이브리드 세그멘테이션."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    if resize_to is not None:
        gray = cv2.resize(gray, resize_to, interpolation=cv2.INTER_AREA)
    h, w = gray.shape
    pixels = gray.reshape(-1, 1).astype(np.float32)

    # 1단계: K-means로 대략적 클러스터링 (논문: "pre-clustering으로 FCM 처리시간 단축")
    km = KMeans(n_clusters=n_clusters, n_init=4, random_state=0).fit(pixels)
    # K-means 중심까지의 거리로 FCM 초기 소속도 행렬 구성 (m=2 → 소속도 ∝ 1/거리²)
    dist = np.abs(pixels - km.cluster_centers_.reshape(1, -1)) + 1e-6
    u_init = 1.0 / dist ** 2
    u_init = (u_init / u_init.sum(axis=1, keepdims=True)).T  # (클러스터 수, 픽셀 수)

    # 2단계: Fuzzy C-means로 정제 (K-means 결과에서 출발 → 무작위 시작보다 빨리 수렴, 매번 같은 결과)
    cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
        pixels.T, c=n_clusters, m=2.0, error=1e-4, maxiter=100, init=u_init)
    membership = np.argmax(u, axis=0)
    lesion_cluster = np.argmin(cntr.flatten())
    mask = (membership == lesion_cluster).reshape(h, w).astype(np.uint8) * 255
    return cv2.bitwise_and(gray, gray, mask=mask)


def resize_for_encoding(img_gray, size):
    return cv2.resize(img_gray, size, interpolation=cv2.INTER_AREA)


def build_dataset(metadata, img_dir, target_class, img_size, max_n=None):
    rows = metadata[metadata["dx"] == target_class]
    if max_n:
        rows = rows.head(max_n)
    imgs = []
    for _, row in rows.iterrows():
        path = find_image_path(row["image_id"])
        if path is None:
            continue
        bgr = cv2.imread(path)
        seg = segment_fcm_kmeans(bgr, n_clusters=4, resize_to=(64, 64))
        imgs.append(resize_for_encoding(seg, (img_size, img_size)).astype(np.float32) / 255.0)
    return np.stack(imgs) if imgs else np.zeros((0, img_size, img_size), dtype=np.float32)

print("세그멘테이션 파이프라인 함수 정의 완료")

In [ ]:
import time

IMG_SIZE = 8  # 2일차/3일차 노트북과 반드시 동일해야 함

df_row = metadata[metadata["dx"] == "df"].iloc[0]
img_bgr = cv2.imread(find_image_path(df_row["image_id"]))
t0 = time.time()
segmented = segment_fcm_kmeans(img_bgr, n_clusters=4, resize_to=(64, 64))
print(f"세그멘테이션 1장 소요시간: {time.time()-t0:.2f}초 (학생 30명이 각자 돌리면 이 시간 x 반복횟수만큼 걸림 — 수업 시간 배분 참고)")

small = resize_for_encoding(segmented, (IMG_SIZE, IMG_SIZE))
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title("원본"); axes[0].axis("off")
axes[1].imshow(segmented, cmap="gray"); axes[1].set_title("세그멘테이션"); axes[1].axis("off")
axes[2].imshow(small, cmap="gray"); axes[2].set_title(f"{IMG_SIZE}x{IMG_SIZE} 축소(양자 인코딩용)"); axes[2].axis("off")
plt.tight_layout(); plt.show()
print(f"필요 데이터 큐빗 수: {int(np.log2(IMG_SIZE*IMG_SIZE))} (단일 서브제너레이터 기준)")

## 5. 전체 DF/NV 전처리 + Drive 저장 (수업 당일 Kaggle 접속 문제 시 폴백용, 선택)

당일 학생들의 Kaggle 접속이 느리거나 실패할 경우를 대비해, 강사가 미리 전처리해둔 데이터를
Drive에 올려두고 공유 링크로 배포할 수 있게 저장합니다. 정상적으로 Kaggle이 되는 학생은
이 파일을 쓸 필요 없이 라이브로 진행하면 됩니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/동아대_QML강의"
os.makedirs(save_dir, exist_ok=True)

df_imgs = build_dataset(metadata, DATA_DIR, "df", IMG_SIZE, max_n=20)
nv_imgs = build_dataset(metadata, DATA_DIR, "nv", IMG_SIZE, max_n=40)
np.save(f"{save_dir}/df_preprocessed_fallback.npy", df_imgs)
np.save(f"{save_dir}/nv_preprocessed_fallback.npy", nv_imgs)
print(f"저장 완료: {save_dir}/df_preprocessed_fallback.npy ({df_imgs.shape}), nv_preprocessed_fallback.npy ({nv_imgs.shape})")
print("(정상 진행 시엔 학생들이 라이브로 만든 df_preprocessed.npy/nv_preprocessed.npy를 쓰므로, 이 파일은 폴백 전용입니다)")

---
## 체크리스트
- [ ] kagglehub 다운로드 정상 (소요시간 확인 — Colab 세션 시작 후 몇 분 걸리는지)
- [ ] EDA 그래프·샘플 이미지 정상 출력
- [ ] 세그멘테이션 1장 소요시간 확인 → 30분 실습 시간 안에 학생들이 반복 실행 가능한지 판단
- [ ] (선택) 폴백 데이터 Drive 저장 완료

**다음 단계**: `day3/prep_day3_train_checkpoints.ipynb`로 넘어가 QGAN·CNN 체크포인트를 생성할 것.
